# Dependency Parsing

![image.png](attachment:image.png)

## Introduction

The language we use on a daily basis operates on the principle of compositionality. This means that the meaning of complex linguistic expressions can be inferred from their component parts and the relationships between them. This property allows language users to be highly creative in constructing utterances while maintaining precision of communication. The way words in a sentence are connected creates a rooted tree structure. The problem we consider in this task is the automatic construction of such trees for Polish sentences. The problem is called syntactic parsing, and specifically we will perform dependency parsing.

Syntactic parsing is generally difficult. For example, although sentences `(1) Maria do jutra jest zajęta.` (Maria is busy until tomorrow) and `(2) Droga do domu jest zajęta.` (The road to the house is occupied) contain the same parts of speech in the same order, and even in exactly the same grammatical form, in sentence (1) the phrase "do jutra" modifies the verb "jest zajęta", while in sentence (2) the phrase "do domu" is a subordinate of the noun "droga". Moreover, sometimes even native speakers can interpret a sentence structure in two different ways: the sentence `Zauważyłem dziś samochód Adama, którego dawno nie widziałem.` (I noticed Adam's car today, whom I hadn't seen in a long time) can be interpreted in two ways depending on whether "którego" refers to "samochód Adama" or "Adama".

There are many different algorithms for solving the dependency parsing problem. Classical methods process a sentence word by word, from left to right, and insert edges based either on a fixed set of rules or a machine learning algorithm. In this task, we will use a different method. Your task will be to predict a dependency tree based on word vectors obtained from the HerBERT model.

HerBERT is a Polish version of BERT, which is a language model that works as follows:
1. BERT has a module called a tokenizer, which divides a sentence into subwords. For example, the sentence `Dostaję klucz i biegnę do swojego pokoju.` is divided into `'Dosta', 'ję', 'klucz', 'i', 'bieg', 'nę', 'do', 'swojego', 'pokoju', '.'`. The tokenizer has a dictionary that assigns unique numbers to subwords: in practice, we get numbers that are not very meaningful to humans: `18577, 2779, 22816, 1009, 4775, 2788, 2041, 5058, 7217, 1899`.
1. Then BERT has a dictionary that converts these numbers into vectors of length 768. Thus, we get a matrix of size `10 x 768`.
1. BERT has 12 layers, each of which takes the result of the previous one and performs a certain transformation on it. The details are not important in this task! What is important, however, is that the entire model is trained automatically using large text corpora. Interpreting the operation of each layer is impossible! However, perhaps in the complex algorithm that BERT learned, different layers play different roles.

## Task

Your task will be automatic syntactic parsing of Polish sentences. We will skip a detailed explanation of how to construct such trees - you can look at the examples yourself! You will receive a training dataset containing 1000 examples of sentence parses. The file `train.conll` contains annotated sentences, for example:

| # | Word      | - | - | - | - | Head | - | - | - |
|---|-----------|---|---|---|---|--------|---|---|---|
| 1 | Wyobraź   | _ | _ | _ | _ | 0      | _ | _ | _ |
| 2 | sobie     | _ | _ | _ | _ | 1      | _ | _ | _ |
| 3 | człowieka | _ | _ | _ | _ | 1      | _ | _ | _ |
| 4 | znajdującego | _ | _ | _ | _ | 3    | _ | _ | _ |
| 5 | się       | _ | _ | _ | _ | 4      | _ | _ | _ |
| 6 | na        | _ | _ | _ | _ | 4      | _ | _ | _ |
| 7 | ogromnej  | _ | _ | _ | _ | 8      | _ | _ | _ |
| 8 | górze     | _ | _ | _ | _ | 6      | _ | _ | _ |
| 9 | .         | _ | _ | _ | _ | 1      | _ | _ | _ |

Which is a way of encoding the following syntactic tree of a compound sentence:
```
      Wyobraź                          
   ______|_____________                 
  |      |         człowieka           
  |      |             |                
  |      |        znajdującego         
  |      |      _______|__________      
  |      |     |                  na   
  |      |     |                  |     
  |      |     |                górze  
  |      |     |                  |     
sobie    .    się              ogromnej
```
We provide you with a Python function for reading examples from this file and visualizing them. Your solution should:
1. Divide the sentence into subwords.
1. Assign a vector to each subword. You should use the final or intermediate vectors computed by the HerBERT model here.
1. Aggregate subword vectors to obtain word vectors.
1. Implement and train a simple model predicting distances in the tree and depths in the tree of individual words in the sentence.
1. Use the distance and depth models to construct a syntactic tree.


## Constraints
- Your final solution will be tested in an environment **without** a GPU.
- Evaluation of your solution (without training) on 200 test examples should take no longer than 5 minutes on Google Colab without a GPU.
- You have at your disposal a BERT-type model: `allegro/herbert-base-cased` and a tokenizer `allegro/herbert-base-cased`. You are not allowed to use other pre-trained models or datasets other than the one provided.
- List of allowed libraries: `transformers`, `nltk`, `torch`.

## Notes and Tips
- Numerous tips can be found in the function templates that you should implement.

## Submission Files
The solution to the task is a zip archive containing:
1. This notebook
2. A file with distance model weights: `distance_model.pth`
3. A file with depth model weights: `depth_model.pth`

Running the entire notebook with the `FINAL_EVALUATION_MODE` flag set to `False` should result in the creation of both weight files in no more than 10 minutes.

## Evaluation
During evaluation, the `FINAL_EVALUATION_MODE` flag will be set to `True`, and then the entire notebook will be run.
Your implemented `parse_sentence` function, whose template you can find at the end of this notebook, will be evaluated on 200 test examples.
The evaluation will be similar to the one implemented in the `evaluate_model` function.
However, note that the final evaluation function will additionally check whether the trees returned by your `parse_sentence` function are valid!

Evaluation cannot take more than 3 minutes. You can run validation of your solution on the provided validation dataset on Google Colab to make sure you are not exceeding the time limit.
Using the `validation_script.py` script, you will be able to make sure your solution will execute correctly on our evaluation servers:

```
python3 validation_script.py --train
python3 validation_script.py
```

During evaluation, we will use two metrics: UUAS and root placement.
1. Root placement means the fraction of examples where you correctly identify the root of the syntactic tree.
2. UUAS for a specific sentence is the fraction of correctly placed edges. UUAS for the dataset is the average of scores for individual sentences.


For this task, you can earn between 0 and 2 points. Your score for this task will be calculated using the function:
```Python
def points(root_placement, uuas):
    def scale(x, lower=0.5, upper=0.85):
        scaled = min(max(x, lower), upper)
        return (scaled - lower) / (upper - lower)
    return (scale(root_placement) + scale(uuas))
```
In other words, your score is the sum of scores for root placement and UUAS. The score for a given metric is 0 if the metric value is below 0.5 and 1 if it is above 0.85. Between these values, the score grows linearly with the metric value.

# Starter Code

In [ ]:
FINAL_EVALUATION_MODE = False  # During evaluation of your solution, we will change this value to True
DEPTH_MODEL_PATH = 'depth_model.pth'  # Do not change!
DISTANCE_MODEL_PATH = 'distance_model.pth'  # Do not change!

In [ ]:
from typing import List

import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import (AutoModel, AutoTokenizer, PreTrainedModel,
                          PreTrainedTokenizer)
from utils import (ListDataset, ParsedSentence, Sentence, merge_subword_tokens,
                   read_conll, uuas_score)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("allegro/herbert-base-cased")
model = AutoModel.from_pretrained("allegro/herbert-base-cased")

In [ ]:
train_sentences = read_conll('train.conll')  # 1000 sentences
val_sentences = read_conll('valid.conll')  # 200 sentences

train_sentences[6].pretty_print()  # display a tree of one sentence
print(train_sentences[6])

# Your Solution

In [ ]:
def get_distances(sentence: ParsedSentence):
    """Find distances between each pair of words in the sentence.
    Returns a numpy array of dimensions (len(sentence), len(sentence))."""

    # TODO: implement me
    ...

    return distances

print(get_distances(train_sentences[1]))

In [ ]:
def get_bert_embeddings(
    sentences_s: List[str],
    tokenizer: PreTrainedTokenizer, 
    model: PreTrainedModel,
    progress_bar: bool = False,
):
    """
    Function returns subword embeddings for a list of sentences.

    Arguments:
        sentences_s: List of sentences. Each sentence is represented as a string.
        tokenizer: HERBERT tokenizer
        model: HERBERT model
        progress_bar: Whether to display a progress bar.

    Returns:
        tokens: A list that for each sentence contains a list of subword tokens of that sentence.
        embeddings: A list that for each sentence contains a list of tensors of dimensions 
            (seq_len, emb_dim). Note that seq_len may be different for different sentences.
    """

    # Tips:
    #  1. You can use the function:
    #   encoded = tokenizer.batch_encode_plus(...)
    #   with torch.no_grad():
    #     model(**encoded, output_hidden_states=True)
    #  2. To speed up computations, remember to group (batch) sentences before passing them to the model.
    #  3. Remember that each sentence can have a different length, so to fill the extra space in the returned
    #   tensor, HERBERT will apply padding. Remember to remove padding from the results.
    #  4. The tokenizer and model use special tokens (e.g., beginning and end of sentence) that should also 
    #   be removed.

    # TODO: implement me
    ...

    return tokens, embeddings

In [ ]:
def get_word_embeddings(sentences: List[Sentence], tokenizer, model):
    """Function returns word embeddings for a list of sentences, using the model and tokenizer."""

    # Tips:
    #  1. Use the get_bert_embeddings function to obtain subword embeddings.
    #  2. Use the merge_subword_tokens function to obtain word embeddings.

    # TODO: implement me
    ...

    return embeddings

In [ ]:
def get_datasets(sentences: List[ParsedSentence], tokenizer, model):
    embeddings = get_word_embeddings(sentences, tokenizer, model)
    distances = [get_distances(sent) for sent in sentences]
    depths = [dist[sent.root][..., None] for dist, sent in zip(distances, sentences)]
    dataset_dist = ListDataset(list(zip(embeddings, distances, sentences)))
    dataset_depth = ListDataset(list(zip(embeddings, depths, sentences)))
    return dataset_dist, dataset_depth


if not FINAL_EVALUATION_MODE:
    trainset_dist, trainset_depth =  get_datasets(train_sentences, tokenizer, model)
    valset_dist, valset_depth = get_datasets(val_sentences, tokenizer, model)

In [ ]:
def pad_arrays(sequence, pad_with=np.inf):
    """
    Assumes that sequence contains arrays (ndarrays) with the same number of dimensions.
    Returns a tensor containing data padded to the same dimensions with the value pad_with, 
    where the sequence index corresponds to the first dimension.
    """

    shapes = np.array([list(seq.shape) for seq in sequence])
    max_lens = list(shapes.max(axis=0))
    padded = [np.pad(
                seq, 
                tuple((0, max_lens[i] - seq.shape[i]) for i in range(seq.ndim)), 
                'constant', 
                constant_values=pad_with
            ) for seq in sequence]
    return torch.tensor(padded)


def collate_fn(batch):
    embeddings, targets, sentences = zip(*batch)
    padded_embeddings = pad_arrays(embeddings, pad_with=0)
    padded_targets = pad_arrays(targets, pad_with=np.inf)
    mask = padded_targets != torch.inf
    return padded_embeddings, padded_targets, mask, sentences


if not FINAL_EVALUATION_MODE:
    dist_trainloader = DataLoader(trainset_dist, batch_size=32, shuffle=True, collate_fn=collate_fn)
    dist_valloader = DataLoader(valset_dist, batch_size=32, shuffle=False, collate_fn=collate_fn)

    depth_trainloader = DataLoader(trainset_depth, batch_size=32, shuffle=True, collate_fn=collate_fn)
    depth_valloader = DataLoader(valset_depth, batch_size=32, shuffle=False, collate_fn=collate_fn)

# dist_trainloader and dist_valloader return tuples (embeddings, distances, masks, sentences)
# depths_trainloader and depths_valloader return tuples (embeddings, depths, masks, sentences)  
# embeddings.shape: (batch_size, max_seq_len, emb_dim)
# distances.shape: (batch_size, max_seq_len, max_seq_len)
# depths.shape: (batch_size, max_seq_len, 1)

In [ ]:
class DistanceModel(torch.nn.Module):
    def __init__(self):
        # TODO: implement me
        ...

    def forward(self, x):
        # TODO: implement me
        ...


class DepthModel(torch.nn.Module):
    def __init__(self):
        # TODO: implement me
        ...

    def forward(self, x):
        # TODO: implement me
        ...

In [ ]:
def loss_fn(output, target, mask):
    # mask, target, output are tensors of the same shape
    # mask contains 1 where target contains data, and 0 where there is padding

    # TODO: implement me
    ...


def train_model(model, dataloader, valloader, epochs, lr):
    """Training loop for your models."""
    # TODO: implement me
    ...


# During evaluation, models should not be retrained.
if not FINAL_EVALUATION_MODE: 
    print("Training depth model")
    depth_model = DepthModel()
    # TODO: set hyperparameters
    train_model(depth_model, depth_trainloader, depth_valloader, lr=..., epochs=...)  
    # save model weights to file
    torch.save(depth_model.state_dict(), DEPTH_MODEL_PATH)

    print("Training distance model")
    distance_model = DistanceModel()
    # TODO: set hyperparameters
    train_model(distance_model, dist_trainloader, dist_valloader, lr=..., epochs=...)
    # save model weights to file
    torch.save(distance_model.state_dict(), DISTANCE_MODEL_PATH)  

In [ ]:
def parse_sentence(sent: Sentence, distance_model, depth_model, tokenizer, model) -> ParsedSentence:
    """Build a syntactic tree for a single sentence.

    Arguments:
        sent: Sentence to parse.
        distance_model: Trained distance model
        depth_model: Trained depth model
        tokenizer: HERBERT tokenizer
        model: HERBERT model

    Returns:
        ParsedSentence: Sentence with predicted syntactic tree.

    """

    # Your solution should:
    # 1. Obtain word embeddings for the sentence.
    # 2. Choose the root of the syntactic tree heuristically, using depth_model.
    # 3. Compute distances between each pair of nodes, using distance_model.
    # 4. Implement a heuristic method of your invention for selecting tree edges 
    #    based on predicted distances.
    # 5. Obtain a ParsedSentence object. You can use the ParsedSentence.from_edges_and_root function
    # 6. Return the ParsedSentence object.

    # Tips:
    #  You can use sent.pretty_print() to visualize the parsed sentence.

    # Note:
    # This function will be used to evaluate your solution. This function should return an actual tree,
    # with len(sent) - 1 edges. If your prediction is not a tree, it will be invalid and
    # you will not earn points for it. If you want to earn only partial points, competing only in the 
    # root placement metric, you should still return a valid tree.

    # TODO: implement me
    ...

if not FINAL_EVALUATION_MODE:
    sent = train_sentences[30]
    parse_sentence(sent, distance_model, depth_model, tokenizer, model).pretty_print()  # Predicted tree
    sent.pretty_print()  # Gold tree (from the dataset)
    print(sent)

# Evaluation
Code very similar to the one below will be used to evaluate the solution on test sentences. By calling the cells below, you can find out how many points your solution would earn if we evaluated it on validation data. Before submitting your solution, make sure the entire notebook runs from beginning to end without errors and without user intervention after running `Run All`.

In [ ]:
def points(root_placement, uuas):
    def scale(x, lower=0.5, upper=0.85):
        scaled = min(max(x, lower), upper)
        return (scaled - lower) / (upper - lower)
    return (scale(root_placement) + scale(uuas))

def evaluate_model(sentences: List[ParsedSentence], distance_model, depth_model, tokenizer, model):
    sum_uuas = 0
    root_correct = 0
    with torch.no_grad():
        for sent in sentences:
            parsed = parse_sentence(sent, distance_model, depth_model, tokenizer, model)
            root_correct += int(parsed.root == sent.root)
            sum_uuas += uuas_score(sent, parsed)
    
    root_placement = root_correct / len(sentences)
    uuas = sum_uuas / len(sentences)

    print(f"UUAS: {uuas * 100:.3}%")
    print(f"Root placement: {root_placement * 100:.3}%")
    print(f"Your score: {points(root_placement, uuas):.1}/2.0")

In [ ]:
if not FINAL_EVALUATION_MODE:
    distance_model_loaded = DistanceModel()
    distance_model_loaded.load_state_dict(torch.load(DISTANCE_MODEL_PATH))

    depth_model_loaded = DepthModel()
    depth_model_loaded.load_state_dict(torch.load(DEPTH_MODEL_PATH))

    evaluate_model(val_sentences, distance_model_loaded, depth_model_loaded, tokenizer, model)